# 5-Model Deep Learning Colorization Comparison

Compares five deep learning colorization variants on the **1,000-image stratified test2017 benchmark** (`data/raw/coco2017/benchmark/`, seed=42). See `docs/benchmark_methodology.md` for the leakage-safe sampling protocol.

## Models
| # | Category | Model |
|---|----------|-------|
| 1 | CNN | Zhang 2016 — **Pretrained** (ECCV weights) |
| 2 | CNN | Zhang 2016 — **Fine-tuned** on COCO 2017 |
| 3 | Interactive CNN | Zhang 2017 SIGGRAPH (auto mode, zero hints) |
| 4 | GAN | DeOldify (NoGAN) |
| 5 | Diffusion | ControlNet + Stable Diffusion 2.1 |

## Metrics
- **PSNR** — pixel-level accuracy (higher is better)
- **SSIM** — structural similarity (higher is better)
- **LPIPS** — learned perceptual distance (lower is better)

All metrics are reported as `mean [95% CI lo, hi]` via percentile bootstrap (`src.deep_learning.stats`, seed=42, n_boot=10,000).

> **Status:** scaffolding. Run `python tools/compare_methods.py` after Phase 3 evaluations to populate the comparison artifacts loaded below.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.deep_learning.utils import load_config

cfg = load_config("../configs/config.yaml")
comparison_dir = os.path.join("..", cfg["paths"]["results_deep"], "comparison")
figures_dir    = os.path.join("..", cfg["paths"]["results_deep"], "figures")
print("comparison_dir =", comparison_dir)
print("figures_dir    =", figures_dir)

## 1. Load comparison results

Expects two artifacts from `tools/compare_methods.py`:
- `comparison_summary.json` — per-model aggregate metrics (mean, std, CI)
- `comparison_metrics.csv` — per-image rows with columns `model`, `image_id`, `psnr`, `ssim`, `lpips`

In [ ]:
summary_path = os.path.join(comparison_dir, "comparison_summary.json")
csv_path     = os.path.join(comparison_dir, "comparison_metrics.csv")

summary, df = None, None
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print(f"Loaded summary: {len(summary)} model entries")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded per-image CSV: {len(df)} rows, models = {sorted(df['model'].unique()) if 'model' in df.columns else 'n/a'}")

if summary is None and df is None:
    print("No comparison results found yet.")
    print("Run Phase 3 evaluations (T3.1-T3.5), then: python tools/compare_methods.py")

## 2. Quantitative metrics table

Per-model mean ± std for PSNR / SSIM / LPIPS, sorted by PSNR (descending).

In [ ]:
def _metrics_table(df):
    if df is None or "model" not in df.columns:
        return None
    agg_cols = [c for c in ("psnr", "ssim", "lpips") if c in df.columns]
    if not agg_cols:
        return None
    table = (
        df.groupby("model")[agg_cols]
          .agg(["mean", "std"])
          .round(4)
          .sort_values(("psnr", "mean"), ascending=False)
          if "psnr" in agg_cols else
        df.groupby("model")[agg_cols].agg(["mean", "std"]).round(4)
    )
    return table

table = _metrics_table(df)
if table is None:
    print("No per-image data — skipping table. (See section 1 to run Phase 4.)")
else:
    print("=== Per-model metrics (mean ± std) ===")
    print(table)

## 3. Bar charts

Three side-by-side panels: PSNR (↑), SSIM (↑), LPIPS (↓). Error bars show std across the 1,000 benchmark images.

In [ ]:
def _bar_charts(df):
    if df is None or "model" not in df.columns:
        return False
    metrics = [m for m in ("psnr", "ssim", "lpips") if m in df.columns]
    if not metrics:
        return False
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4))
    if len(metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, metrics):
        agg = df.groupby("model")[metric].agg(["mean", "std"]).sort_values("mean", ascending=(metric == "lpips"))
        ax.bar(agg.index, agg["mean"], yerr=agg["std"], capsize=4)
        ax.set_title(f"{metric.upper()} {'(lower is better)' if metric == 'lpips' else '(higher is better)'}")
        ax.set_xlabel("model")
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    plt.show()
    return True

if not _bar_charts(df):
    print("No per-image data — skipping bar charts.")

## 4. Qualitative grid

A precomputed visual grid is expected at `results/deep_learning/figures/qualitative_grid.png` (one row per benchmark image, one column per model). Generated by `tools/compare_methods.py`.

In [ ]:
qual_path = os.path.join(figures_dir, "qualitative_grid.png")
if os.path.exists(qual_path):
    img = plt.imread(qual_path)
    fig, ax = plt.subplots(figsize=(14, max(6, img.shape[0] / 80)))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Qualitative grid (one row per image, one column per model)")
    plt.show()
else:
    print(f"No qualitative grid at {qual_path}.")
    print("Generate via: python tools/compare_methods.py --max-images 50")

## Next steps

Once Phase 4 has run, this notebook will be expanded with:
- Per-model analysis sections (Zhang16 Pretrained vs Fine-tuned saturation analysis, Zhang17 auto-mode behaviour, DeOldify vs ControlNet trade-offs)
- Speed vs quality scatter (inference time vs PSNR)
- Failure-case gallery
- Trade-off discussion mirroring `reports/deep_learning/sections/discussion.tex`

See `tasks/todo.md` and `tasks/plan.md` for the full pipeline.